> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Synthetic — mvGWAS canonical xarray result

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr
import vigor
print("VIGOR",vigor.__version__)
cfg=vigor.simulation_scenario("complex"); cfg.update(dict(heritability=.55,compute_grms=True))
# Genotypes live in a reusable 2-bit packed Zarr store (GT_packed2); nothing below holds a
# dense animals x variants matrix.
DATA=Path("vigor_tutorial_data"); DATA.mkdir(exist_ok=True)
GENO=str(DATA/"complex_n300_m800_chr4.zarr")
sim=vigor.simulate_packed_cohort(GENO,n_samples=300,n_variants=800,n_chromosomes=4,n_traits=4,seed=42,genotype_kwargs={"generations":100},trait_kwargs=cfg)
Y=sim.phenotypes.to_pandas(); Y.index=Y.index.astype(str)
# mvGWAS streams the store block by block: LOCO kernel and G/E per chromosome. The default
# engine='fast' keeps the exact score X'Py and calibrates its information per chromosome
# (engine='exact' evaluates it for every variant). Additive+dominance tests use the PLINK
# pipeline route, vigor.mvGWAS(bfile, pheno, model='add-dom').
gwas=vigor.mvGWAS(Y,genotypes=GENO,chromosomes=["1"])
print(gwas)
print("dims",dict(gwas.sizes),"variables",list(gwas.data_vars)[:8])
print("fast calibration:",gwas.attrs.get("scan_group_metadata"))
top=gwas.neglog_p.max("trait",skipna=True).sortby(gwas.neglog_p.max("trait",skipna=True),ascending=False).isel(variant=slice(0,5))
print(top)
# Projected discovery scan (engine='fast' with state=): one GEMM per block against the projected
# exact dual, calibrated with simulated null traits so real signal cannot deflate it. In real scans
# build one null per chromosome on its LOCO kernel and pass a {chromosome: state} mapping.
from vigor import mvgwas_fast
from vigor.gwas.scan import covariance_matrices
K=sim.grms.sel(kernel="additive").to_pandas(); K.index=K.index.astype(str); K.columns=K.columns.astype(str)
G,E=covariance_matrices(vigor.genetic_varcov_PSD(K,Y,rank_G="full",rank_E="full",analytic_se=False,progress=False))
state=mvgwas_fast.state_from_vigor_null(vigor.prepare_score_null(Y,K,G,E),rank=2,null_reps=4)
fast=vigor.mvGWAS(GENO,engine="fast",state=state,chromosomes=["1"],sample_ids=list(Y.index))
print("fast scan:",fast.diagnostics["n_variants_scanned"],"variants; lambda",fast.calibrated_states["1"].calibration.lambda_gc)
print(fast.top_variants)
